# FutStats — YOLO training on Google Colab (T4 GPU)

Trains a futsal **player + ball** detector on the cleaned dataset
(`futstats_dataset_clean/` — 348 train / 50 val / 100 test, classes `0: player`, `1: ball`).

## Before running
1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU.**
2. Upload the dataset zip at the "Upload dataset" step below.
   - It was created on your Mac at `data/futstats_dataset_clean.zip`.
   - To recreate it yourself: `cd data && zip -r futstats_dataset_clean.zip futstats_dataset_clean`

In [ ]:
# 1) Confirm the T4 GPU is attached
!nvidia-smi

In [ ]:
# 2) Install Ultralytics YOLO
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3) Upload the dataset zip
Run the next cell, then choose `futstats_dataset_clean.zip` from your Mac.

> **Large upload / flaky connection?** Put the zip on Google Drive instead and mount it:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> !cp "/content/drive/MyDrive/futstats_dataset_clean.zip" /content/
> ```
> then skip the `files.upload()` cell and go straight to unzip.

In [ ]:
# 3) Upload futstats_dataset_clean.zip   (skip if you copied it from Drive)
from google.colab import files
uploaded = files.upload()

In [ ]:
# 4) Unzip into /content/
!unzip -q -o futstats_dataset_clean.zip -d /content/
!ls /content/futstats_dataset_clean

In [ ]:
# 5) Rewrite data.yaml for Colab paths
#    (the zipped data.yaml points at a macOS path — overwrite it for this machine)
import os, yaml
ROOT = "/content/futstats_dataset_clean"
cfg = {
    "path":  ROOT,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "names": {0: "player", 1: "ball"},
}
with open(f"{ROOT}/data.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(f"{ROOT}/data.yaml").read())

In [ ]:
# 6) Sanity check: image/label counts per split (expect 348 / 50 / 100)
import glob
for s in ["train", "valid", "test"]:
    ni = len(glob.glob(f"{ROOT}/{s}/images/*.jpg"))
    nl = len(glob.glob(f"{ROOT}/{s}/labels/*.txt"))
    print(f"{s:5}: {ni} images / {nl} labels")

In [ ]:
# 7) Train on the T4 (device=0)
from ultralytics import YOLO

MODEL = "yolov8n.pt"   # nano fits the T4 at imgsz=1280; try "yolov8s.pt" for more accuracy
model = YOLO(MODEL)

model.train(
    data="/content/futstats_dataset_clean/data.yaml",
    epochs=100,
    imgsz=1280,
    device=0,
    # batch=8,   # uncomment if the T4 OOMs at imgsz=1280 (or batch=-1 for auto-batch)
)

In [ ]:
# 8) Evaluate on the held-out TEST split
metrics = model.val(split="test")
print("mAP50-95:", round(metrics.box.map, 3))
print("mAP50   :", round(metrics.box.map50, 3))

In [ ]:
# 9) Download the trained weights (best.pt) to your Mac
from google.colab import files
best = model.trainer.best          # path to this run's best.pt
print("best weights:", best)
files.download(str(best))
# Tip: save it in your repo as models/futsal.pt so the src/*.py scripts pick it up.

## Done
- **Weights:** `best.pt` (downloaded above) → put it at `models/futsal.pt` in the repo.
- **Diagnostics:** `runs/detect/train/` has `results.png` (curves), `confusion_matrix.png`,
  and `val_batch*_pred.jpg` (sample predictions) — open them in the Colab file browser.
- **Predict locally:**
  `from ultralytics import YOLO; YOLO("models/futsal.pt").predict("videos/match1.mov", save=True)`